# EDA — Amazon Reviews 2023 (+ Goodreads)

**Goal:** understand the data well enough that retrieval, prompts, and evaluation are designed for what *actually* exists — not what we wish existed.

Each section answers a concrete question the rest of the pipeline depends on:

| Section | Question it answers | Who needs it |
|---|---|---|
| A | What columns / nulls do we have? | everyone |
| B | How long is a typical review? | embeddings (chunk size) |
| C | How skewed is the rating distribution? | eval (need balanced metrics?) |
| D | How many users have ≥ 2 reviews? | **make-or-break** for "predict last review" task |
| E | How many reviews per product? | retrieval (do we have enough context per item?) |
| F | What time range does the dataset cover? | possibly filter to recent reviews only |
| G | What does the vocabulary look like? | text cleaning before embedding |
| H | How do Books vs Electronics differ? | category-specific prompt tuning |

The deliverable is the **final "Implications" cell** — the plots are evidence for those conclusions.

In [ ]:
"""Setup — run me first."""
import sys, pathlib

# Make `app.*` importable from inside notebooks/
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from app.core import config
from app.core import constants as C

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("Project root:", config.ROOT_DIR)
print("Raw data dir:", config.DATA_RAW_DIR)
print("Categories:  ", config.AMAZON_CATEGORIES)

## Loading the data

Two paths — use whichever your wifi allows:

- **A.** Files already downloaded by `app.data.loader` → parse them with the teammate's `parse_amazon_category`. This returns DataFrames already mapped to the unified schema.
- **B.** Stream a sample directly from HuggingFace. No full download. The raw HF columns differ from the unified schema, so cell B renames them at the end.

Run **one** of the two cells below.

In [ ]:
"""Scenario A — data already on disk (run app/data/loader.py first)."""
from app.data.loader import find_amazon_files
from app.data.preprocess import parse_amazon_category

paths = find_amazon_files(["Books"])
reviews_df, items_df = parse_amazon_category(
    paths["Books"]["reviews"],
    paths["Books"]["meta"],
    category="Books",
)

print("reviews:", reviews_df.shape)
print("items:  ", items_df.shape)
reviews_df.head(3)

In [ ]:
"""Scenario B — stream a sample from HuggingFace (no full download).

Use when wifi can't handle the full file. Pulls SAMPLE_SIZE rows and persists
them as a Parquet under data/raw/ so we don't re-stream every time.
"""
from datasets import load_dataset

SAMPLE_SIZE = 50_000
sample_path = config.DATA_RAW_DIR / "books_reviews_sample.parquet"

if sample_path.exists():
    reviews_df = pd.read_parquet(sample_path)
    print(f"Loaded cached sample → {len(reviews_df):,} rows")
else:
    ds = load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023",
        "raw_review_Books",
        split="full",
        streaming=True,
        trust_remote_code=True,
    )

    rows = []
    for i, r in enumerate(ds):
        if i >= SAMPLE_SIZE:
            break
        rows.append(r)
    reviews_df = pd.DataFrame(rows)

    # Map HF column names → unified schema so downstream cells work either way.
    reviews_df = reviews_df.rename(columns={
        "text":            C.F_REVIEW_TEXT,
        "rating":          C.F_RATING,
        "user_id":         C.F_USER_ID,
        "parent_asin":     C.F_ITEM_ID,
        "timestamp":       C.F_TIMESTAMP,
    })
    reviews_df[C.F_CATEGORY] = "Books"
    reviews_df[C.F_DOMAIN]   = C.DOMAIN_AMAZON

    sample_path.parent.mkdir(parents=True, exist_ok=True)
    reviews_df.to_parquet(sample_path, index=False)
    print(f"Streamed + cached {len(reviews_df):,} rows → {sample_path.name}")

# items_df is optional in Scenario B — most cells only need reviews_df.
items_df = pd.DataFrame()
reviews_df.head(3)

## Section A — Schema & nulls

What columns do we have, and where is data missing?

In [ ]:
reviews_df.info()

missing_pct = (reviews_df.isna().mean() * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
missing_pct.plot.barh(ax=ax, color="#c0392b")
ax.set_title("% missing per column")
ax.set_xlabel("% missing")
plt.tight_layout()
plt.show()

missing_pct

## Section B — Review length distribution

Tells the embeddings team what chunk size to pick. BGE-small handles ~512 tokens; if 95% of reviews are under 200 words, no chunking is needed.

In [ ]:
reviews_df["word_count"] = (
    reviews_df[C.F_REVIEW_TEXT].fillna("").str.split().str.len()
)

fig, ax = plt.subplots(figsize=(9, 4))
reviews_df["word_count"].clip(0, 1000).hist(bins=60, ax=ax, color="#2980b9")
ax.set_title("Review length (words, clipped at 1000)")
ax.set_xlabel("words")
ax.set_ylabel("# reviews")
plt.tight_layout()
plt.show()

reviews_df["word_count"].describe(percentiles=[.5, .9, .95, .99]).round(1)

## Section C — Rating distribution

Amazon reviews are notoriously 5-star skewed. If the imbalance is severe, MAE alone is misleading — we'd want per-class accuracy or a balanced sampler for evaluation.

In [ ]:
rating_counts = reviews_df[C.F_RATING].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
rating_counts.plot.bar(ax=ax, color="#27ae60")
ax.set_title("Rating distribution")
ax.set_xlabel("rating")
ax.set_ylabel("# reviews")
plt.tight_layout()
plt.show()

share = (rating_counts / rating_counts.sum() * 100).round(1)
print("Share per rating (%):")
print(share)

## Section D — Reviews per user  ⚠️ make-or-break

The whole "predict the user's next review" task requires users with **≥ 2 reviews** (one to learn their style from, one to predict). With `HOLDOUT_LAST_N = 1` we need every test user to have at least 2.

If most users only ever wrote 1 review, the project pivots — surface this finding immediately.

In [ ]:
per_user = reviews_df.groupby(C.F_USER_ID).size()

needed = config.HOLDOUT_LAST_N + 1
viable_users = (per_user >= needed).sum()
total_users  = len(per_user)

print(f"Total users:                  {total_users:,}")
print(f"Users with ≥ {needed} reviews:        {viable_users:,}  "
      f"({viable_users/total_users*100:.1f}%)")
print(f"Users meeting MIN_USER_REVIEWS={config.MIN_USER_REVIEWS}: "
      f"{(per_user >= config.MIN_USER_REVIEWS).sum():,}")

fig, ax = plt.subplots(figsize=(9, 4))
per_user.clip(upper=50).hist(bins=50, ax=ax, color="#8e44ad")
ax.set_yscale("log")
ax.set_title("Reviews per user (clipped at 50, log y)")
ax.set_xlabel("# reviews")
ax.set_ylabel("# users (log)")
plt.tight_layout()
plt.show()

per_user.describe(percentiles=[.5, .75, .9, .95, .99]).round(1)

## Section E — Reviews per item

How many reviews exist per product? Affects retrieval: items with too few reviews are bad RAG targets.

In [ ]:
per_item = reviews_df.groupby(C.F_ITEM_ID).size()

print(f"Total items:                       {len(per_item):,}")
print(f"Items with ≥ {config.MIN_ITEM_REVIEWS} reviews (threshold): "
      f"{(per_item >= config.MIN_ITEM_REVIEWS).sum():,}")

fig, ax = plt.subplots(figsize=(9, 4))
per_item.clip(upper=100).hist(bins=50, ax=ax, color="#e67e22")
ax.set_yscale("log")
ax.set_title("Reviews per item (clipped at 100, log y)")
ax.set_xlabel("# reviews")
ax.set_ylabel("# items (log)")
plt.tight_layout()
plt.show()

per_item.describe(percentiles=[.5, .75, .9, .95, .99]).round(1)

## Section F — Temporal distribution

The 2023 dataset spans ~25 years. Do we restrict to recent years for relevance, or use everything?

In [ ]:
# Timestamps may be unix seconds OR milliseconds depending on which loader path
# was used. preprocess._ts_from_amazon converts to seconds; HF stream gives ms.
ts = reviews_df[C.F_TIMESTAMP].astype("Int64")
unit = "ms" if ts.dropna().max() > 1_000_000_000_000 else "s"
reviews_df["date"] = pd.to_datetime(ts, unit=unit, errors="coerce")

by_year = reviews_df["date"].dt.year.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
by_year.plot.bar(ax=ax, color="#16a085")
ax.set_title("Reviews by year")
ax.set_xlabel("year")
ax.set_ylabel("# reviews")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Date range: {reviews_df['date'].min()}  →  {reviews_df['date'].max()}")

## Section G — Vocabulary / top terms

Surface obvious text-cleaning needs before embedding (e.g. heavy HTML residue, mojibake, spam patterns).

In [ ]:
import re
from collections import Counter

# Light English stopword list — avoids pulling sklearn just for this.
STOPWORDS = {
    "the","and","this","that","with","have","from","were","they","their","them",
    "what","when","your","you","but","for","not","are","was","has","had","its",
    "than","then","into","just","very","also","been","more","some","such","only",
    "much","most","over","because","there","these","those","about","would","could",
    "should","like","really","one","two","get","got","still","even","make","made",
    "back","good","great","time","book","read","story","author","character","characters",
}

sample_n = min(5_000, len(reviews_df))
words = Counter()
for txt in reviews_df[C.F_REVIEW_TEXT].dropna().sample(sample_n, random_state=42):
    for w in re.findall(r"\b[a-z]{4,}\b", txt.lower()):
        if w not in STOPWORDS:
            words[w] += 1

top = pd.Series(dict(words.most_common(30)))
fig, ax = plt.subplots(figsize=(8, 8))
top.plot.barh(ax=ax, color="#34495e")
ax.invert_yaxis()
ax.set_title(f"Top 30 content words (sample of {sample_n:,} reviews)")
ax.set_xlabel("count")
plt.tight_layout()
plt.show()

## Section H — Books vs Electronics comparison

Run sections A–G again with Electronics loaded, then plot side-by-side. Same code, different category in the loader call. Stub below — fill in once Electronics is downloaded / streamed.

In [ ]:
# Pre-generated by: python scripts/run_eda.py (after fetch_samples.py)
from IPython.display import Image, display

display(Image(filename="outputs/H_books_vs_electronics.png"))
print("Full comparison plot saved under notebooks/outputs/")

## Implications for the pipeline

_Re-run `python scripts/fetch_samples.py` + `python scripts/run_eda.py`. Implications live in `outputs/eda_summary.json`._

- **Embedding chunk size (§B):** Books 95th percentile ~589 words (Electronics ~202). Truncate or chunk around **400 words** for BGE; Electronics reviews are shorter on average.
- **Trainable users (§D):** Books **56.6%** of users (210/371) have ≥2 reviews in sample; Electronics **63%**. Task is viable on streamed sample — confirm on full data after `MIN_USER_REVIEWS` filtering.
- **Rating skew (§C):** ~65% five-star in both categories. Use MAE **and** per-rating metrics / macro-F1 in eval.
- **Temporal scope (§F):** 2000–2023. Consider **2015+** filter for demo if old reviews add noise.
- **Books vs Electronics (§H):** See `outputs/H_books_vs_electronics.png`. Electronics reviews are shorter; tune prompts per category.

**Architect:** embed per-review text; aggregate at retrieval. Contract: `app/team_contract.py` → `predict_next_review(user_id, category)`.